# 02 — Train a climbing-hold detector (run this on Google Colab)

**BoulderVision Phase 1 — fine-tuning.** Trains a single-purpose YOLOv8 model
that detects **climbing holds + volumes**. Color classification is NOT learned
here — our HSV classifier (`src/utils.py`) colors each detected hold locally.

## Why Colab?
Training on CPU is impractically slow. Colab gives you a free GPU.
**Before running:** `Runtime` → `Change runtime type` → Hardware accelerator → **GPU (T4)**.

## What you need ready
1. A Roboflow **Private API key** (Settings → API Keys). You'll paste it into a
   hidden prompt below — it is never stored in this notebook.
2. Nothing else — the dataset downloads automatically.

## Steps in this notebook
1. Confirm GPU → 2. Install deps → 3. Download dataset → 4. Inspect classes →
5. Train → 6. Evaluate → 7. Download `best.pt` to use locally.

## 1. Confirm a GPU is attached
If this errors or shows no GPU, set Runtime → Change runtime type → GPU first.

In [ ]:
!nvidia-smi

## 2. Install dependencies (in the Colab VM, not your machine)

In [ ]:
!pip install -q ultralytics roboflow

## 3. Download the climbing-holds dataset
Uses the public **Climbing Holds and Volumes** dataset. You'll be prompted for
your API key via a hidden field — paste it there, press Enter. It is not saved
into the notebook, so this notebook is safe to keep in a public repo.

In [ ]:
from getpass import getpass
from roboflow import Roboflow

# Hidden prompt — paste your Roboflow Private API key and press Enter.
api_key = getpass("Roboflow Private API key: ")

# These three identify the public dataset; change only if you switch datasets.
WORKSPACE = "robworkspace-xfv3r"
PROJECT = "climbing-holds-and-volumes-c91sa"
VERSION = 1

rf = Roboflow(api_key=api_key)
project = rf.workspace(WORKSPACE).project(PROJECT)
dataset = project.version(VERSION).download("yolov8")
print("\nDataset downloaded to:", dataset.location)

## 4. Inspect the dataset
See how many classes and images we have. Expect classes like `hold` / `volume`.

In [ ]:
import yaml, glob, os

data_yaml = os.path.join(dataset.location, "data.yaml")
with open(data_yaml) as f:
    d = yaml.safe_load(f)

print("Classes:", d["names"])
print("Num classes:", d.get("nc", len(d["names"])))
for split in ("train", "valid", "test"):
    n = len(glob.glob(os.path.join(dataset.location, split, "images", "*")))
    print(f"  {split}: {n} images")

## 5. Train
Start from pretrained `yolov8s` (transfer learning) — a good speed/accuracy
balance. On a T4, ~100 epochs takes roughly 1–2 hours depending on dataset
size. `patience=20` stops early if validation stops improving.

Tips: drop `epochs` to 30 for a quick first run; switch to `yolov8n` for speed
or `yolov8m` for more accuracy if the T4 has headroom.

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8s.pt")  # pretrained COCO weights as a starting point
results = model.train(
    data=data_yaml,
    epochs=100,
    imgsz=640,
    batch=16,
    patience=20,
    project="bouldervision",
    name="holds_yolov8s",
)

## 6. Evaluate
`mAP50` is the headline number (mean average precision at IoU 0.5). For a usable
hold detector you're hoping for ~0.6+; higher is better.

In [ ]:
metrics = model.val()
print("mAP50-95:", round(float(metrics.box.map), 3))
print("mAP50:   ", round(float(metrics.box.map50), 3))

## 7. (Optional) Sanity-check on one of your own gym photos
Upload a photo from your gym to see real detections before downloading the model.

In [ ]:
from google.colab import files
from PIL import Image

uploaded = files.upload()  # pick one climbing-wall photo
for fname in uploaded:
    res = model(fname, conf=0.25)[0]
    print(f"{fname}: {len(res.boxes)} holds detected")
    res.save(filename="pred_" + fname)
    display(Image.open("pred_" + fname))

## 8. Download the trained weights
Saves `best.pt` to your computer. Then, **locally in the BoulderVision repo**:

1. Move the file to `models/holds.pt`.
2. In `config/settings.yaml` set: `hold_detector: "models/holds.pt"`.
3. Run: `python -m src.hold_detector data/input/<your_photo>.jpeg`

The rest of the pipeline (color classification, drawing) is unchanged.

In [ ]:
from google.colab import files

best = "bouldervision/holds_yolov8s/weights/best.pt"
print("Downloading:", best)
files.download(best)